# 13장 경쟁위험 분석 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 17장의 가상 코호트(만성콩팥병 4기 고령 환자 3,500명, 투석과 투석 전 사망)로 사건 열, Aalen–Johansen 누적발생함수, 1 − Kaplan-Meier와의 차이, Gray 검정, 원인별 위험비, Fine–Gray 하위분포 위험비, 비례위험 검정을 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages("cmprsk")를 실행합니다.


## a. 사건 열과 누적발생함수

### 셀 1. 자료 불러오기와 사건 열 (17장 a 절)

In [ ]:
suppressPackageStartupMessages({library(survival); library(cmprsk)})
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ckd_competing.csv")     # 한 줄 = 환자 한 명
str(d)
table(drug = d$drug, status = d$status)  # 0 = 중도절단, 1 = 투석, 2 = 투석 전 사망
d$ev <- factor(d$status, levels = 0:2,
               labels = c("censor", "dialysis", "death"))   # 첫 수준이 중도절단

### 셀 2. 청구 날짜로 사건 열 만들기 (17장 b 절)

In [ ]:
x <- data.frame(id = 1:4, index = as.Date("2018-01-01"),
                dial  = as.Date(c("2019-03-01", NA, "2020-06-30", NA)),           # 투석 시작일
                death = as.Date(c(NA, "2018-11-20", "2020-06-30", NA)),           # 사망일
                cens  = as.Date(c("2022-12-31", "2022-12-31", "2022-12-31", "2021-05-31")))  # 중도절단일
end <- pmin(x$dial, x$death, x$cens, na.rm = TRUE)            # 가장 이른 날 = 추적 종료일
x$years  <- round(as.numeric(end - x$index) / 365.25, 3)
x$status <- ifelse(!is.na(x$dial) & x$dial == end, 1,         # 같은 날이면 투석을 먼저 확인
            ifelse(!is.na(x$death) & x$death == end, 2, 0))
x[, c("id", "years", "status")]

### 셀 3. 두 군의 누적발생함수 (17장 a 절)

In [ ]:
fit <- survfit(Surv(years, ev) ~ drug, data = d)      # 군마다 Aalen–Johansen 추정
s <- summary(fit, times = c(1, 3, 5))
p <- s$pstate; colnames(p) <- c("free", "dialysis", "death")
data.frame(drug = sub("drug=", "", s$strata), time = s$time, round(p, 4),
           dial_lo = round(s$lower[, 2], 4), dial_up = round(s$upper[, 2], 4),
           death_lo = round(s$lower[, 3], 4), death_up = round(s$upper[, 3], 4))

### 셀 4. 표 17-2: 1 − Kaplan-Meier와 누적발생함수 (17장 a 절)

In [ ]:
b  <- d[d$drug == "B", ]
km <- summary(survfit(Surv(years, status == 1) ~ 1, data = b), times = 1:5)   # 사망을 중도절단한 1 − KM
aj <- summary(survfit(Surv(years, ev) ~ 1, data = b), times = 1:5)
cbind(year = 1:5, round(cbind(one_minus_km = 1 - km$surv, dialysis = aj$pstate[, 2], death = aj$pstate[, 3],
                               free = aj$pstate[, 1], total = rowSums(aj$pstate)) * 100, 1))
bad <- survfit(Surv(years, status) ~ 1, data = b)    # 0/1/2를 숫자 그대로 넣으면
c(n = bad$n, events = sum(bad$n.event))

### 셀 5. 5년 누적발생률의 차이 (17장 a 절)

In [ ]:
s5 <- summary(fit, times = 5)
p  <- s5$pstate[, 2]; se <- s5$std.err[, 2]            # 5년 투석 누적발생률과 표준오차(A, B)
dif <- unname(p[1] - p[2]); sed <- sqrt(sum(se^2))
round(c(A = p[1], B = p[2], diff = dif, lower = dif - 1.96 * sed, upper = dif + 1.96 * sed) * 100, 1)
round(s5$pstate[, 3] * 100, 1)                           # 5년 투석 전 사망(A, B)

### 셀 6. Gray 검정과 로그순위법 (17장 a 절)

In [ ]:
ci <- cuminc(d$years, d$status, group = d$drug, cencode = 0)
ci$Tests                                   # Gray 검정: 1행 = 투석, 2행 = 투석 전 사망
survdiff(Surv(years, status == 1) ~ drug, data = d)   # 사망을 중도절단한 로그순위법

## b. 원인별 위험비와 하위분포 위험비

### 셀 7. 표 17-3: 군별 속도 (17장 b 절)

In [ ]:
agg <- aggregate(cbind(py = years, dialysis = status == 1, death = status == 2) ~ drug, data = d, sum)
cbind(agg, rate_dialysis = round(agg$dialysis / agg$py * 100, 2),
      rate_death = round(agg$death / agg$py * 100, 2))   # 100인년당

### 셀 8. 원인별 위험비 (17장 b 절)

In [ ]:
d$A <- as.integer(d$drug == "A")
ci3 <- function(fit) round(summary(fit)$conf.int[, c(1, 3, 4)], 3)
cs_dial  <- coxph(Surv(years, status == 1) ~ A, data = d)   # 투석만 사건, 사망은 중도절단
cs_death <- coxph(Surv(years, status == 2) ~ A, data = d)   # 사망만 사건, 투석은 중도절단
rbind(dialysis = ci3(cs_dial), death = ci3(cs_death))
summary(cs_dial)$coefficients

### 셀 9. 하위분포 위험비: 예정일 방법, finegray, crr (17장 b 절)

In [ ]:
d$t_sd <- ifelse(d$status == 2, d$max_years, d$years)       # 사망자를 추적 종료 예정 시점까지 남김
fg_date <- coxph(Surv(t_sd, status == 1) ~ A, data = d)
fgd  <- finegray(Surv(years, ev) ~ A + id, data = d, etype = "dialysis")   # 중도절단 확률로 가중해 남김
fg_w <- coxph(Surv(fgstart, fgstop, fgstatus) ~ A, weights = fgwt, cluster = id, data = fgd)
fg_crr <- crr(d$years, d$status, cov1 = cbind(A = d$A), failcode = 1, cencode = 0)
rbind(date_method = ci3(fg_date), finegray = ci3(fg_w),
      crr = round(summary(fg_crr)$conf.int[, c(1, 3, 4)], 3))
nrow(fgd)                                                    # finegray가 만든 줄 수
fg_row <- coxph(Surv(fgstart, fgstop, fgstatus) ~ A, weights = fgwt, data = fgd)   # cluster = id를 빼면
round(c(cluster_id = sqrt(fg_w$var[1, 1]), by_row = sqrt(fg_row$var[1, 1]), crr = sqrt(fg_crr$var[1, 1])), 4)

### 셀 10. 비례위험 검정 (17장 b 절)

In [ ]:
round(rbind(dialysis = cox.zph(cs_dial)$table["A", ], death = cox.zph(cs_death)$table["A", ],
            subdist_date = cox.zph(fg_date)$table["A", ]), 3)

### 셀 11. 가중 누적발생함수의 신뢰구간 (17장 b 절)

In [ ]:
b$w <- 2                                              # 모든 환자의 가중치를 2로
cif5 <- function(f) {
  s <- summary(f, times = 5)
  round(c(cif = s$pstate[2], lower = s$lower[2], upper = s$upper[2]), 4)
}
rbind(no_weight = cif5(survfit(Surv(years, ev) ~ 1, data = b)),
      weight_2  = cif5(survfit(Surv(years, ev) ~ 1, data = b, weights = w, id = id)))

## c. 과제 정답

### 셀 12. 과제 1 정답. 요양병원 입소자 8명

In [ ]:
small <- data.frame(month = 1:8, ev = factor(c(1, 2, 2, 1, 2, 1, 0, 0), 0:2, c("censor", "fracture", "death")))
km6 <- summary(survfit(Surv(month, ev == "fracture") ~ 1, data = small), times = 6)
aj6 <- summary(survfit(Surv(month, ev) ~ 1, data = small), times = 6)
round(c(one_minus_km = 1 - km6$surv, cif_fracture = aj6$pstate[2], cif_death = aj6$pstate[3], free = aj6$pstate[1]), 3)

### 셀 13. 과제 2 정답. 고관절 골절 자료로 동료의 분석 다시 보기

In [ ]:
fx <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/dementia_fracture.csv")
fx$ev <- factor(fx$status, 0:2, c("censor", "fracture", "death")); fx$C <- as.integer(fx$drug == "C")
sf <- summary(survfit(Surv(years, ev) ~ drug, data = fx), times = 5)
km <- summary(survfit(Surv(years, status == 1) ~ drug, data = fx), times = 5)
round(cbind(fracture = sf$pstate[, 2], death = sf$pstate[, 3], one_minus_km = 1 - km$surv) * 100, 1)
fgx <- finegray(Surv(years, ev) ~ C, data = fx, etype = "fracture")
rbind(cs_fracture = ci3(coxph(Surv(years, status == 1) ~ C, data = fx)),
      cs_death    = ci3(coxph(Surv(years, status == 2) ~ C, data = fx)),
      subdist     = ci3(coxph(Surv(fgstart, fgstop, fgstatus) ~ C, weights = fgwt, data = fgx)))